# DrugReview: the annotation release

This notebook reproduces the manuscript's description of the annotation (Table 2 and the text of Section 4.1), its repeatability on the 2,000-review repeated subset (the repeatability paragraph and Supplementary Table S2), the descriptive facts about the class-score stream given in the notes to Table 3, and the share of reviews whose model input fits in the 200-token window (Section 3.3). It reads the frozen release and the repeat files written by the labelling notebook; it calls no model and fits nothing. The evidence passages are verbatim review text, so only counts, lengths, and agreement statistics are printed. Results are written to `00_annotation_summary/outputs/`.

The corpus flow (Supplementary Table S1) is produced by `00_labelling/DrugReview_CleanComments.ipynb`; the entropy of the class-score vector and the score stream's repeatability are reported by `00_entropy/DrugReview_Entropy_Analysis.ipynb`.

## 0. Setup: paths and constants

In [1]:
import os, json, glob, warnings
from pathlib import Path
import numpy as np, pandas as pd
from sklearn.metrics import f1_score, cohen_kappa_score
warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 60); pd.set_option("display.max_rows", 200)

# ROOT is the folder that holds 00_labelling/, 00_split/, 01_OriginalLabel/, ... (Drive: NLP_Projects/03_DrugReview; local: "01. Colab backup").
if os.path.isdir("/content") and not os.environ.get("DRUGREVIEW_ROOT"):
    from google.colab import drive; drive.mount("/content/drive", force_remount=False)

def resolve_project_root():
    override = os.environ.get("DRUGREVIEW_ROOT")
    if override:
        candidate = Path(override).expanduser().resolve()
        if not candidate.is_dir():
            raise FileNotFoundError(f"DRUGREVIEW_ROOT does not exist: {candidate}")
        return candidate
    for base in [Path.cwd(), *Path.cwd().parents, Path("/content/drive/MyDrive/NLP_Projects/03_DrugReview")]:
        for candidate in (base, base / "DrugReview_new"):
            if (candidate / "01_OriginalLabel").is_dir():
                return candidate.resolve()
    raise FileNotFoundError("Set DRUGREVIEW_ROOT to the folder containing 01_OriginalLabel, 00_split, and the run data.")

ROOT = resolve_project_root()
OUT = Path(os.environ.get("DRUGREVIEW_ANNOTATION_OUT", ROOT / "00_annotation_summary" / "outputs")); OUT.mkdir(parents=True, exist_ok=True)

PROTOCOL_ID = "v2_ef356fdf87"
REL = ROOT / "00_labelling" / "output_v2" / PROTOCOL_ID
CLASSES = ["VERY_NEGATIVE", "NEGATIVE", "NEUTRAL", "POSITIVE", "VERY_POSITIVE"]
A3 = ["NEGATIVE", "NEUTRAL", "POSITIVE"]
ASPECTS = ["efficacy", "safety", "burden", "cost"]
ENCODERS = ("albert", "biobert")
RUNS = {"03": "03_Full_aspects/outputs/aspect_auxiliary_v2_3class",
        "04": "04_Holistic_all_aspects/outputs/aspect_auxiliary_v2_3class",
        "A0": "ablation/A0_Matched_holistic_control/outputs/aspect_auxiliary_v2_3class",
        "5p": "05_Confirmatory/outputs/multilevel_representation_only_v2",
        "5r": "05_Confirmatory/outputs/multilevel_representation_residual_v2",
        "5j": "05_Confirmatory/outputs/multilevel_probability_bottleneck_v2",
        "5d": "05_Confirmatory/outputs/multilevel_detached_probability_bottleneck_v2",
        "6ff": "06_Semi_confirmatory/outputs/semi_4anchored_4free_v2",
        "6a": "06_Semi_confirmatory/outputs/semi_4anchored_0free_v2",
        "6f": "06_Semi_confirmatory/outputs/semi_0anchored_4free_v2",
        "7e": "07_Evidence_hierarchical/outputs/evidence_hierarchical_v2_evidence_1",
        "7c": "07_Evidence_hierarchical/outputs/evidence_hierarchical_v2_evidence_0"}
NAMES = {"03": "3 aspect baseline", "04": "4 parallel", "A0": "4 no aspect heads", "5p": "5 fusion", "5r": "5 residual",
         "5j": "5 bottleneck, joint", "5d": "5 bottleneck, detached", "6ff": "6 4 anchored + 4 free", "6a": "6 4 anchored + 0 free",
         "6f": "6 0 anchored + 4 free", "7e": "7 evidence loss on", "7c": "7 evidence loss off"}

def test_predictions(run, enc, usecols=None):
    df = pd.read_csv(ROOT / RUNS[run] / enc / f"{enc}_test_predictions.csv", usecols=usecols)
    return df.sort_values("uniqueID").reset_index(drop=True)

# Narrative-group bootstrap on group-level confusion matrices: each draw resamples the narrative groups of the test partition
# with replacement (numpy default_rng(seed).integers(0, G, G)), so reviews that share a narrative stay together, and every
# model scored on the same draws is paired. 95% percentile intervals.
def confusion_metrics(M):
    """M: (..., K, K) counts, rows = true. Returns weighted F1, macro F1, quadratic-weighted kappa."""
    K = M.shape[-1]; W = np.array([[(i - j) ** 2 for j in range(K)] for i in range(K)], float)
    tp = np.einsum("...ii->...i", M); rows = M.sum(-1); cols = M.sum(-2); n = rows.sum(-1, keepdims=True)
    den = 2 * tp + (cols - tp) + (rows - tp); f1 = np.where(den > 0, 2 * tp / np.where(den > 0, den, 1), 0.0)
    wf1 = (f1 * rows).sum(-1) / n[..., 0]; mf1 = f1.mean(-1)
    E = rows[..., :, None] * cols[..., None, :] / n[..., None]; qwk = 1 - (W * M).sum((-1, -2)) / (W * E).sum((-1, -2))
    return wf1, mf1, qwk

def group_draws(groups, n, seed):
    codes = pd.factorize(pd.Series(groups))[0]; G = int(codes.max() + 1); rng = np.random.default_rng(seed)
    counts = np.empty((n, G))                              # filled in place: 10,000 x ~19,000 draws stay near 1.5 GB
    for i in range(n):
        counts[i] = np.bincount(rng.integers(0, G, G), minlength=G)
    return codes, G, counts

def boot_metrics(y, preds, groups, K, n=1000, seed=2025):
    """preds: dict name -> predicted ids (aligned with y). Returns dict name -> (point (wf1, mf1, qwk), draws (n, 3))."""
    codes, G, cnt = group_draws(groups, n, seed); y = np.asarray(y); out = {}
    for k, p in preds.items():
        C = np.zeros((G, K * K)); np.add.at(C, (codes, y * K + np.asarray(p)), 1.0)
        out[k] = (np.array(confusion_metrics(C.sum(0).reshape(K, K))), np.stack(confusion_metrics((cnt @ C).reshape(n, K, K)), 1))
    return out

def ci(a, axis=0):
    return np.percentile(a, [2.5, 97.5], axis=axis)

def fmt_ci(x, lo, hi, d=3):
    return f"{x:.{d}f} ({lo:.{d}f}, {hi:.{d}f})"

def save(df, name, **kw):
    df.to_csv(OUT / name, **kw); print(f"written: outputs/{name}")

print("ROOT:", ROOT); print("OUT :", OUT)

Mounted at /content/drive
ROOT: /content/drive/MyDrive/NLP_Projects/03_DrugReview
OUT : /content/drive/MyDrive/NLP_Projects/03_DrugReview/00_annotation_summary/outputs


## 1. What the release contains

Panel A compares the holistic label with the writer's rating binned into five classes (1–2, 3–4, 5–6, 7–8, 9–10). Panel B gives the five-way state of each aspect, and Panel C the number of evidence passages returned per review. The cells after the table give the agreement of the two holistic labels, the aspects mentioned, and the length and direction of the passages.

In [2]:
# The frozen release: holistic classes of the binned rating and of the annotation, the five aspect states, and the passages.
# Only aggregates are printed; the passages are verbatim review text and are never shown.
import re
STATES = ["POSITIVE", "NEGATIVE", "MIXED", "UNMENTIONED", "UNCLEAR"]
cols = ["uniqueID", "rating", "ai_sent5_hard"] + [f"ai_{a}_{s}" for a in ASPECTS for s in ("state", "evidence")]
rel = pd.read_csv(REL / f"drugscom_plus_ai_labels_{PROTOCOL_ID}.csv", usecols=cols)
rel["rating5"] = pd.cut(rel.rating, [0, 2, 4, 6, 8, 10], labels=CLASSES).astype(str)
print(f"reviews: {len(rel):,}")
pa = pd.DataFrame({"binned_rating_pct": rel.rating5.value_counts(normalize=True), "annotation_pct": rel.ai_sent5_hard.value_counts(normalize=True)}).reindex(CLASSES).mul(100).round(1)
print("\nPanel A: holistic classes (% of reviews)\n" + pa.T.to_string())
pb = pd.DataFrame({a: rel[f"ai_{a}_state"].value_counts(normalize=True).reindex(STATES, fill_value=0) for a in ASPECTS}).T.mul(100).round(1)
print("\nPanel B: aspect states (% of reviews)\n" + pb.to_string())
passages = {a: rel[f"ai_{a}_evidence"].map(lambda s: json.loads(s) if isinstance(s, str) else []) for a in ASPECTS}
pc = pd.Series({a: passages[a].map(len).mean() for a in ASPECTS}); pc["all"] = sum(passages[a].map(len) for a in ASPECTS).mean()
print("\nPanel C: passages per review (mean)\n" + pc.round(2).to_frame("mean").T.to_string())
save(pd.concat({"holistic": pa.T, "aspect_states": pb, "passages_per_review": pc.to_frame("mean").T}, axis=0), "release_composition.csv")

# The binned rating against the holistic label.
y_r = rel.rating5.map(CLASSES.index).values; y_h = rel.ai_sent5_hard.map(CLASSES.index).values
print(f"\nexact agreement {100 * (y_r == y_h).mean():.1f}% | within one class {100 * (np.abs(y_r - y_h) <= 1).mean():.1f}% | QWK {cohen_kappa_score(y_r, y_h, weights='quadratic'):.3f}")
ct = pd.crosstab(rel.rating5, rel.ai_sent5_hard, normalize="index").reindex(index=CLASSES, columns=CLASSES).mul(100).round(1)
print("rows = binned rating, columns = holistic label (% of the row)\n" + ct.to_string())
# Aspects mentioned, the neutral training class, and the passages.
mentioned = pd.DataFrame({a: rel[f"ai_{a}_state"] != "UNMENTIONED" for a in ASPECTS})
print("\nmentioned (% of reviews): " + ", ".join(f"{a} {100 * mentioned[a].mean():.1f}" for a in ASPECTS) + f" | aspects mentioned per review: {mentioned.sum(1).mean():.2f}")
print("share of each aspect's neutral training class (mixed + unmentioned + unclear) that is mixed: "
      + ", ".join(f"{a} {pb.loc[a, 'MIXED'] / pb.loc[a, ['MIXED', 'UNMENTIONED', 'UNCLEAR']].sum():.2f}" for a in ASPECTS))
flat = [(a, q, d) for a in ASPECTS for lst in passages[a] for q, d in lst]
P = pd.DataFrame(flat, columns=["aspect", "passage", "direction"]); P["words"] = P.passage.map(lambda s: len(re.findall(r"\w+", s)))
print(f"\npassages returned: {len(P):,} | per review {len(P) / len(rel):.2f}")
print("mean words per passage: " + ", ".join(f"{a} {P[P.aspect == a].words.mean():.1f}" for a in ASPECTS))
print("direction (% of passages): " + ", ".join(f"{k.lower()} {v:.1f}" for k, v in (P.direction.value_counts(normalize=True) * 100).items()))

reviews: 102,061

Panel A: holistic classes (% of reviews)
                   VERY_NEGATIVE  NEGATIVE  NEUTRAL  POSITIVE  VERY_POSITIVE
binned_rating_pct           17.0       7.3      9.4      18.4           47.9
annotation_pct              18.1      16.8      3.2      30.1           31.7

Panel B: aspect states (% of reviews)
          POSITIVE  NEGATIVE  MIXED  UNMENTIONED  UNCLEAR
efficacy      63.4      19.4   13.7          1.3      2.2
safety        14.9      50.9    7.8         25.7      0.7
burden         9.7      40.0    4.9         42.6      2.8
cost           2.4       4.6    0.5         92.3      0.2

Panel C: passages per review (mean)
      efficacy  safety  burden  cost   all
mean      1.92     1.2    0.83  0.09  4.05
written: outputs/release_composition.csv

exact agreement 58.1% | within one class 91.9% | QWK 0.828
rows = binned rating, columns = holistic label (% of the row)
ai_sent5_hard  VERY_NEGATIVE  NEGATIVE  NEUTRAL  POSITIVE  VERY_POSITIVE
rating5               

## 2. Repeatability under the frozen protocol

The holistic label and the aspect states are read from the replication summary; the passages are compared review by review. Exact: the replication returns the same passage after whitespace and case normalization. Near: exact, or a passage that contains, is contained in, or shares at least half of its word types with it. Direction is compared on exactly matched passages.

In [3]:
# Repeatability under the frozen protocol: 2,000 training reviews annotated a second time. The holistic label and the aspect states
# are read from the replication summary written by the labelling notebook; the passages are compared here, review by review.
stab = pd.read_csv(REL / "stability_repeated_subset.csv"); stab = stab[~stab.field.str.startswith("fused")]   # holistic label and aspect fields only
print(stab.to_string(index=False))
st = stab.set_index("field")
print(f"\nholistic label reproduced for {100 * st.loc['holistic (5 classes)', 'agreement']:.2f}% (kappa {st.loc['holistic (5 classes)', 'kappa']:.3f}, QWK {st.loc['holistic (5 classes)', 'quadratic kappa']:.3f}); "
      + "aspect states: " + ", ".join(f"{a} {100 * st.loc[f'{a} state (5)', 'agreement']:.2f}% (kappa {st.loc[f'{a} state (5)', 'kappa']:.3f})" for a in ASPECTS))
def load_part(f): return pd.DataFrame([json.loads(l) for l in open(REL / "repeat" / "parts" / f)])
rep = load_part("parsed_20260920T062903_e258b8.jsonl"); assert (rep.task == "aspect_rep").all(); rep["uniqueID"] = rep.uniqueID.astype(int)
norm = lambda s: re.sub(r"\s+", " ", str(s)).strip().lower().strip('"\'.,;:! ')
def parse(x):
    try: v = json.loads(x) if isinstance(x, str) else x
    except Exception: return []
    return [(norm(q), d) for q, d in v if isinstance(q, str)] if isinstance(v, list) else []
tok = lambda s: set(re.findall(r"\w+", s))
ma = rep[rep.status == "OK"].merge(rel[["uniqueID"] + [f"ai_{a}_{s}" for a in ASPECTS for s in ("state", "evidence")]], on="uniqueID", validate="one_to_one")
rows = []
for x in ASPECTS:
    e1, e2 = ma[f"ai_{x}_evidence"].map(parse), ma[f"{x}_evidence"].map(parse); both = (e1.map(len) > 0) | (e2.map(len) > 0)
    n_any = int(both.sum()); ident = 0; jac = []; exact = near = tot = 0; dir_ok = dir_n = 0
    for q1, q2 in zip(e1[both], e2[both]):
        s1 = {q for q, _ in q1}; s2 = {q for q, _ in q2}; ident += s1 == s2; jac.append(len(s1 & s2) / len(s1 | s2) if (s1 | s2) else 1.0); d2 = {q: d for q, d in q2}
        for q, dd in q1:
            tot += 1
            if q in s2: exact += 1; dir_n += 1; dir_ok += d2[q] == dd
            if q in s2 or any((q in r) or (r in q) or (len(tok(q) & tok(r)) / max(len(tok(q) | tok(r)), 1) >= 0.5) for r in s2): near += 1
    rows.append({"aspect": x, "state_agreement": round(float((ma[f"ai_{x}_state"] == ma[f"{x}_state"]).mean()), 4), "reviews_with_passages": n_any,
                 "same_set": round(ident / max(n_any, 1), 3), "jaccard": round(float(np.mean(jac)), 3), "exact": round(exact / max(tot, 1), 3),
                 "near": round(near / max(tot, 1), 3), "direction_on_exact": round(dir_ok / max(dir_n, 1), 3)})
s2 = pd.DataFrame(rows); print("\npassages, production run against the replication\n" + s2.to_string(index=False)); save(s2, "passage_repeatability.csv", index=False)

                  field    n  agreement  kappa  quadratic kappa
   holistic (5 classes) 2000     0.9895 0.9859           0.9967
     efficacy state (5) 2000     0.9675 0.9375              NaN
efficacy vote (-1/0/+1) 2000     0.9685 0.9383              NaN
       safety state (5) 2000     0.9595 0.9381              NaN
  safety vote (-1/0/+1) 2000     0.9610 0.9357              NaN
       burden state (5) 2000     0.8995 0.8427              NaN
  burden vote (-1/0/+1) 2000     0.9090 0.8391              NaN
         cost state (5) 2000     0.9930 0.9495              NaN
    cost vote (-1/0/+1) 2000     0.9940 0.9529              NaN

holistic label reproduced for 98.95% (kappa 0.986, QWK 0.997); aspect states: efficacy 96.75% (kappa 0.938), safety 95.95% (kappa 0.938), burden 89.95% (kappa 0.843), cost 99.30% (kappa 0.950)

passages, production run against the replication
  aspect  state_agreement  reviews_with_passages  same_set  jaccard  exact  near  direction_on_exact
efficacy       

## 3. The class-score stream

How the scores come back before they are normalized: whether they sum to one, whether they are reported in tenths, and whether the returned label is the class with the largest score.

In [4]:
# The class-score stream: how the returned scores look before normalization, and how often the returned label is the top class.
sc = pd.read_csv(REL / "scores" / "drugscom_holistic_scores_s_7c599892d6.csv", usecols=lambda c: c in ["uniqueID", "scores_status", "scores_attempt", "scores_raw_sum", "scores_label_is_argmax"] + [f"ai_s_{c.lower()}" for c in CLASSES])
S = sc[[f"ai_s_{c.lower()}" for c in CLASSES]].to_numpy(float)
print(f"reviews: {len(sc):,} | status: {sc.scores_status.value_counts().to_dict()} | answered at attempt: {sc.scores_attempt.value_counts().to_dict()}")
print(f"returned scores sum to one: {100 * (np.abs(sc.scores_raw_sum - 1) < 1e-9).mean():.2f}% (minimum sum {sc.scores_raw_sum.min():.2f})")
print(f"all five scores are multiples of 0.1: {100 * np.all(np.isclose(S * 10, np.round(S * 10)), axis=1).mean():.1f}%")
print(f"returned label is the class with the largest score: {100 * sc.scores_label_is_argmax.astype(bool).mean():.1f}%")
top = np.isclose(S, S.max(1)[:, None]); print(f"two classes tied at the largest score: {100 * (top.sum(1) > 1).mean():.1f}%; most classes tied: {top.sum(1).max()}")

reviews: 102,061 | status: {'OK': 102061} | answered at attempt: {1: 102061}
returned scores sum to one: 99.96% (minimum sum 0.90)
all five scores are multiples of 0.1: 95.5%
returned label is the class with the largest score: 99.8%
two classes tied at the largest score: 13.9%; most classes tied: 2


## 4. Length of the model input

The encoders read at most 200 tokens. The model input is the review after the cleaning of the rung notebooks (lowercased, URLs and user mentions replaced by placeholder tokens, punctuation removed), tokenized with the ALBERT and BioBERT tokenizers saved with the rung-7 runs; the count includes the special tokens.

In [5]:
# Share of reviews whose model input fits in the 200-token window of the encoders. clean_text is the cleaning used by the rung notebooks.
import re
from transformers import AutoTokenizer
def clean_text(text):
    if pd.isna(text): return ""
    text = text.lower()
    text = re.sub(r"http\S+", " urltoken ", text)
    text = re.sub(r"@\w+", " usertoken ", text)
    text = re.sub(r"#(\w+)", r" hashtag_\1 ", text)
    text = re.sub(r"[^\w\s]", "", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()
texts = pd.read_csv(REL / f"drugscom_plus_ai_labels_{PROTOCOL_ID}.csv", usecols=["review"]).review.map(clean_text).tolist()
rows = []
for enc in ("albert", "biobert"):
    tok = AutoTokenizer.from_pretrained(ROOT / "07_Evidence_hierarchical" / "outputs" / "evidence_hierarchical_v2_evidence_1" / enc / "tokenizer")
    n = np.array([len(ids) for ids in tok(texts, add_special_tokens=True, truncation=False)["input_ids"]])
    rows.append({"encoder": enc, "reviews": len(n), "median_tokens": int(np.median(n)), "p95_tokens": int(np.percentile(n, 95)), "within_200_pct": round(100 * (n <= 200).mean(), 1)})
tl = pd.DataFrame(rows); print("model input length (tokens, with special tokens)\n" + tl.to_string(index=False)); save(tl, "model_input_length.csv", index=False)

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (665 > 512). Running this sequence through the model will result in indexing errors


model input length (tokens, with special tokens)
encoder  reviews  median_tokens  p95_tokens  within_200_pct
 albert   102061            116         164            99.8
biobert   102061            120         169            99.7
written: outputs/model_input_length.csv
